# Teaching a student

While a student works on a training puzzle, a teacher who knows the optimal moves (breadth-first search) can step in (`scripts/teach.py`). Each teacher acts with a probability:

- **random p** - on each turn, with probability p, shows the next optimal move, which the student then plays (p = 1: the student only watches - pure imitation).
- **correct after mistake q** - every move the student makes stands; after one that wasn't optimal, with probability q, the teacher points the mistake out and shows the optimal move from the board it led to, on the student's next turn.
- **flag mistakes q** - after a move that wasn't optimal, with probability q, the teacher points the mistake out; it never shows a move.

Pointing out a mistake tells the student that move was wrong on that board, and nothing more: the student never learns it as a target there and is pushed away from it. A move the teacher says nothing about means nothing. Each intervention - a move shown, or a mistake pointed out - is one unit of the teaching budget; a mistake pointed out together with the correction right after it counts once.

The student is **pure intuition**: its network picks a move every turn, sampled from its move probabilities, avoiding its last few boards; no search. It learns by expert iteration: each round it attempts every training puzzle with its teacher - **fresh puzzles every round** by default, like new homework - then trains on what the attempts showed (its solved attempts with loops cut out, the moves the teacher showed, the mistakes it pointed out) and starts the next round from the result. After each round it is **tested without a teacher** on held-out puzzles, behaving as it does while learning, a few attempts per puzzle.

Experiments run from the command line with `scripts/teach_experiment.py` (one configuration per run, saved under `models/teach/<run>/`); this notebook plots saved runs. Different teaching probabilities spend very different budgets, so give each its own run and plot them separately.

In [ ]:
import sys
sys.path.append("../scripts")
import os

import matplotlib.pyplot as plt

import teach_experiment as te

## Running experiments

From `scripts/`, one run per configuration (see `python teach_experiment.py --help`). Runs use every CPU core and the GPU, so start one at a time - e.g. chained with `&&`, in the background with a log:

```
nohup sh -c "python teach_experiment.py fresh_p0.1 --teachers random:0.1 correct-after:0.1 flag:0.1 && \
             python teach_experiment.py fresh_p1 --teachers random:1 correct-after:1 flag:1" > teach.log 2>&1 &
```

More examples:

- long-term: `python teach_experiment.py fresh_p0.1 --rounds 10 --resume` continues a saved run to 10 rounds;
- harder puzzles: `--train-bins 1-8 9-12 13-16 17-20 21-30 --test-bins 21-30 31-40` (moves per attempt are capped at `--step-factor` x the puzzle's optimal length, at least `--min-steps`);
- sparser data: `--train-per-bin 25`;
- the same training puzzles every round: `--reuse-puzzles`;
- re-test a saved run with other test settings: `--test-only --test-runs 5`.

A run can also be started from here (it blocks the notebook until done):

In [ ]:
RUN_HERE = False
if RUN_HERE:
    te.main(["fresh_p0.1", "--teachers", "random:0.1", "correct-after:0.1", "flag:0.1"])

## Runs to plot

In [ ]:
RUNS = ["fresh_p0.1", "fresh_p1"]  # saved runs, each plotted on its own
COLORS = {"random": "C0", "correct-after": "C1", "flag": "C2", "correct-before": "C3"}  # by kind of teacher

runs = {name: te.load_run(name) for name in RUNS}
for name, run in runs.items():
    c = run["config"]
    print(f"{name}: teachers {', '.join(c['teachers'])}; {c['rounds']} rounds of {c['train_per_bin']} puzzles per bin "
          f"{c['train_bins']} ({'reused' if c['reuse_puzzles'] else 'fresh'} each round), {c['attempts']} attempts each; "
          f"tested on {c['test_per_bin']} puzzles per bin {c['test_bins']}, {c['test_runs']} attempts each "
          f"at temperature {c['test_temperature']}")

## Training

Per round: training attempts solved, moves the teacher showed, mistakes it pointed out, its interventions, boards the student learned from, and its validation agreement (with that round's own attempts) at the epoch it kept; for the mistake teachers, what the mistakes they noticed were like.

In [ ]:
for name, run in runs.items():
    print(f"##### {name}")
    for teacher, entry in run["teachers"].items():
        print(f"=== {teacher} ===")
        for r in entry["history"]:
            print(f"  round {r['round']}: solved {r['solved']}/{r['attempts']} training attempts, "
                  f"{r['taught_moves']} of {r['moves']} moves shown, {r['flagged']} mistakes pointed out, "
                  f"{r['interventions']} interventions, {r['boards']} boards learned from, "
                  f"validation agreement {r['agreement']:.0%} (epoch {r['best_epoch']})")
            counts = r["teacher_counts"]
            if "mistakes_worse" in counts or "mistakes_sideways" in counts:
                mistakes = counts.get("mistakes_worse", 0) + counts.get("mistakes_sideways", 0)
                line = (f"    {mistakes} mistakes noticed: {counts.get('mistakes_worse', 0)} took the board further "
                        f"from the goal, {counts.get('mistakes_sideways', 0)} sideways")
                if counts.get("corrections"):
                    line += (f"; undoing was optimal for {counts.get('undo_optimal', 0) / counts['corrections']:.0%} "
                             f"of corrections, and the move shown was the undo for "
                             f"{counts.get('shown_undo', 0) / counts['corrections']:.0%}")
                print(line)

## Learning trajectories

Each teacher's student after every round (round 0 = untrained), tested without a teacher. Each puzzle's test attempts are averaged first, since attempts on the same puzzle aren't independent; error bands are standard errors across puzzles, and moves / optimal is over solved attempts.

The bottom panels show how much each teacher intervened - per move the student played in training, and as the teaching budget: each point is a round's tested solve rate, placed at the number of interventions on training attempts up to then.

In [ ]:
def band(ax, xs, stats, color, label):
    """Means with a standard-error band."""
    ms = [m for m, _ in stats]
    ses = [se if se == se else 0.0 for _, se in stats]
    ax.plot(xs, ms, marker="o", color=color, label=label)
    ax.fill_between(xs, [m - e for m, e in zip(ms, ses)], [m + e for m, e in zip(ms, ses)],
                    color=color, alpha=0.2, linewidth=0)


trajectories = {}  # (run, teacher) -> [(round, summary)]
for name, run in runs.items():
    c = run["config"]
    for (teacher, entry), spec in zip(run["teachers"].items(), c["teachers"]):
        trajectories[(name, teacher)] = [(r, te.summarize(entry["results"][r], c["test_runs"], te.bins(c["test_bins"])))
                                         for r in sorted(entry["results"])]
    print(f"{name}:")
    for teacher in run["teachers"]:
        last_round, s = trajectories[(name, teacher)][-1]
        (rate, se), (ratio, ratio_se) = s["solve rate"], s["moves / optimal"]
        print(f"  {teacher:<28} after round {last_round}: solved {rate:.0%} ± {se:.0%}, "
              f"{ratio:.1f} ± {ratio_se:.1f}x optimal")

os.makedirs("../visualization", exist_ok=True)
for name, run in runs.items():
    c = run["config"]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    (ax_rate, ax_moves), (ax_share, ax_budget) = axes
    for (teacher, entry), spec in zip(run["teachers"].items(), c["teachers"]):
        color = COLORS[spec.partition(":")[0]]
        points = trajectories[(name, teacher)]
        rounds = [r for r, _ in points]
        for ax, key in ((ax_rate, "solve rate"), (ax_moves, "moves / optimal")):
            band(ax, rounds, [s[key] for _, s in points], color, teacher)
        history = entry["history"]
        ax_share.plot([h["round"] for h in history], [h["interventions"] / h["moves"] for h in history],
                      marker="o", color=color, label=teacher)
        budget = [0]
        for h in history:
            budget.append(budget[-1] + h["interventions"])
        by_round = dict(points)
        ses = [by_round[r]["solve rate"][1] for r in rounds]
        ax_budget.errorbar([budget[r] for r in rounds], [by_round[r]["solve rate"][0] for r in rounds],
                           yerr=[se if se == se else 0.0 for se in ses], marker="o", capsize=3, color=color,
                           label=teacher)
    ax_rate.set_ylabel("Solve rate (tested without teacher)")
    ax_rate.set_ylim(0, 1.05)
    ax_moves.set_ylabel("Moves / optimal (solved test attempts)")
    ax_moves.axhline(1.0, color="gray", linestyle="--")
    for ax in (ax_rate, ax_moves):
        ax.set_xlabel("Round (0 = untrained)")
        ax.set_xticks(range(c["rounds"] + 1))
    ax_share.set_ylabel("Interventions per training move")
    ax_share.set_ylim(-0.05, 1.05)
    ax_share.set_xlabel("Round")
    ax_share.set_xticks(range(1, c["rounds"] + 1))
    ax_budget.set_ylabel("Solve rate (tested without teacher)")
    ax_budget.set_ylim(0, 1.05)
    ax_budget.set_xlabel("Teaching budget: interventions during training so far")
    ax_rate.legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.savefig(f"../visualization/teaching_{name}.png")
    plt.show()

The same trajectories by puzzle difficulty: solve rate (tested without teacher) per test distance bin, one figure per run.

In [ ]:
for name, run in runs.items():
    c = run["config"]
    test_bins = te.bins(c["test_bins"])
    fig, axes = plt.subplots(1, len(test_bins), figsize=(3.4 * len(test_bins), 3.8), sharey=True, squeeze=False)
    for ax, (lo, hi) in zip(axes[0], test_bins):
        for (teacher, entry), spec in zip(run["teachers"].items(), c["teachers"]):
            points = trajectories[(name, teacher)]
            band(ax, [r for r, _ in points], [s[(lo, hi)] for _, s in points], COLORS[spec.partition(":")[0]], teacher)
        ax.set_title(f"distance {lo}-{hi}")
        ax.set_xlabel("Round")
        ax.set_xticks(range(c["rounds"] + 1))
        ax.set_ylim(0, 1.05)
    axes[0][0].set_ylabel("Solve rate")
    axes[0][0].legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.savefig(f"../visualization/teaching_{name}_by_distance.png")
    plt.show()